In [13]:
# 0) Setup
!pip -q install numpy pandas scikit-learn torch matplotlib ta einops

import os, time, math, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import ta
from einops import rearrange

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [14]:
# 1) Data Preprocessing
import pandas as pd
import numpy as np

df = pd.read_csv("btc_2y_1h.csv")
df = df.loc[:, ~df.columns.duplicated()]

rename_map = {}
for c in df.columns:
    cl = c.lower()
    if cl in ["open", "high", "low", "close", "volume", "log_return"]:
        rename_map[c] = cl
    elif cl in ["time", "timestamp", "date", "datetime"]:
        rename_map[c] = "time"
df = df.rename(columns=rename_map)
df = df.loc[:, ~df.columns.duplicated()]

if "time" in df.columns:
    col = df["time"]
    if isinstance(col, pd.DataFrame):
        col = col.iloc[:, 0]
    if pd.api.types.is_numeric_dtype(col):
        unit = "ms" if float(col.max()) > 1e12 else "s"
        df["time"] = pd.to_datetime(col, unit=unit, utc=True)
    else:
        df["time"] = pd.to_datetime(col, utc=True, errors="coerce")

if "time" in df.columns:
    df = df.dropna(subset=["time"]).sort_values("time").reset_index(drop=True)

base_cols = ["open", "high", "low", "close", "volume"]
df = df.dropna(subset=[c for c in base_cols if c in df.columns]).reset_index(drop=True)

if "log_return" not in df.columns and "close" in df.columns:
    df["log_return"] = np.log(df["close"]).diff()


In [15]:
# 2) Target Generation
def future_sum(series, k):
    s = series.reset_index(drop=True)
    return s.shift(-1).rolling(k).sum()

df["y_2h"] = future_sum(df["log_return"], 2)
df["y_4h"] = future_sum(df["log_return"], 4)
df["y_6h"] = future_sum(df["log_return"], 6)

df = df.dropna(subset=["y_2h", "y_4h", "y_6h"]).reset_index(drop=True)

TARGETS = ["y_2h", "y_4h", "y_6h"]
feature_cols = [c for c in df.columns if c not in ["time"] + TARGETS]


In [16]:
# 3)
if "time" in df.columns and pd.api.types.is_datetime64_any_dtype(df["time"]):
    t_train_end = pd.Timestamp("2025-05-31 23:59:59", tz="UTC")
    t_val_end   = pd.Timestamp("2025-08-31 23:59:59", tz="UTC")
    t_test_end  = pd.Timestamp("2025-10-15 23:59:59", tz="UTC")
    df_train = df[df["time"] <= t_train_end]
    df_val   = df[(df["time"] > t_train_end) & (df["time"] <= t_val_end)]
    df_test  = df[(df["time"] > t_val_end) & (df["time"] <= t_test_end)]
    if len(df_val) < 500 or len(df_test) < 500:
        N = len(df)
        df_train = df.iloc[:int(N * 0.8)]
        df_val   = df.iloc[int(N * 0.8):int(N * 0.9)]
        df_test  = df.iloc[int(N * 0.9):]
else:
    N = len(df)
    df_train = df.iloc[:int(N * 0.8)]
    df_val   = df.iloc[int(N * 0.8):int(N * 0.9)]
    df_test  = df.iloc[int(N * 0.9):]


In [17]:
# 4) Data Split

WINDOW = 240
BATCH = 256

scaler = StandardScaler().fit(df_train[feature_cols].values)

def make_xy_multi(df_split, window=168):
    Xf = scaler.transform(df_split[feature_cols].values)
    Yf = df_split[TARGETS].values
    X_list, Y_list = [], []
    for i in range(window, len(Xf)):
        X_list.append(Xf[i-window:i, :])
        Y_list.append(Yf[i])
    X = np.array(X_list, dtype=np.float32)
    Y = np.array(Y_list, dtype=np.float32)
    return X, Y

X_tr, Y_tr = make_xy_multi(df_train, WINDOW)
X_va, Y_va = make_xy_multi(df_val, WINDOW)
X_te, Y_te = make_xy_multi(df_test, WINDOW)

class TSDS(Dataset):
    def __init__(self, X, Y):
        self.X = torch.from_numpy(X)
        self.Y = torch.from_numpy(Y)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.Y[i]

tr_ds = TSDS(X_tr, Y_tr)
va_ds = TSDS(X_va, Y_va)
te_ds = TSDS(X_te, Y_te)

train_dl = DataLoader(tr_ds, batch_size=BATCH, shuffle=True, drop_last=True)
val_dl = DataLoader(va_ds, batch_size=BATCH, shuffle=False)
test_dl = DataLoader(te_ds, batch_size=BATCH, shuffle=False)

in_feats = X_tr.shape[-1]



In [18]:
# 5) Model 1 - RNN(GRU/LSTM)

class RNNMulti(nn.Module):
    def __init__(self, in_feats, hidden=64, layers=2, kind="gru", dropout=0.2, out_dim=3, bidir=False):
        super().__init__()
        self.kind = kind
        if kind == "gru":
            self.rnn = nn.GRU(in_feats, hidden, num_layers=layers,
                              batch_first=True, dropout=dropout if layers > 1 else 0, bidirectional=bidir)
        else:
            self.rnn = nn.LSTM(in_feats, hidden, num_layers=layers,
                               batch_first=True, dropout=dropout if layers > 1 else 0, bidirectional=bidir)
        last_dim = hidden * (2 if bidir else 1)
        self.head = nn.Sequential(
            nn.Linear(last_dim, last_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(last_dim, out_dim)
        )

    def forward(self, x):
        out, _ = self.rnn(x)
        h = out[:, -1, :]
        y = self.head(h)
        return y


In [19]:
# 6) Model 2 - TCN

class CausalConv1d(nn.Conv1d):
    def __init__(self, in_ch, out_ch, k, dilation=1):
        super().__init__(in_ch, out_ch, k, padding=(k - 1) * dilation, dilation=dilation)
        self.left_trim = (k - 1) * dilation

    def forward(self, x):
        out = super().forward(x)
        return out[..., :-self.left_trim] if self.left_trim > 0 else out


class TCNBlock(nn.Module):
    def __init__(self, ch_in, ch_out, k=3, dilation=1, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            CausalConv1d(ch_in, ch_out, k, dilation),
            nn.ReLU(),
            nn.Dropout(dropout),
            CausalConv1d(ch_out, ch_out, k, dilation),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.res = nn.Conv1d(ch_in, ch_out, 1) if ch_in != ch_out else nn.Identity()

    def forward(self, x):
        y = self.net(x)
        r = self.res(x)
        return y + r


class TCNMulti(nn.Module):
    def __init__(self, in_feats, ch=64, depth=4, k=3, dropout=0.2, out_dim=3):
        super().__init__()
        layers = []
        c_in = in_feats
        for i in range(depth):
            layers += [TCNBlock(c_in, ch, k=k, dilation=2 ** i, dropout=dropout)]
            c_in = ch
        self.tcn = nn.Sequential(*layers)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(ch, ch),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(ch, out_dim)
        )

    def forward(self, x):
        x = rearrange(x, 'b w f -> b f w')
        z = self.tcn(x)
        y = self.head(z)
        return y



In [20]:
# 7) Model 3 - Transformer Encoder(lite)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        L = x.size(1)
        return x + self.pe[:, :L, :]


class TransEncMulti(nn.Module):
    def __init__(self, in_feats, d_model=64, nhead=4, num_layers=2, dim_ff=128, dropout=0.1, out_dim=3):
        super().__init__()
        self.inproj = nn.Linear(in_feats, d_model)
        self.pos = PositionalEncoding(d_model)
        enc_layer = nn.TransformerEncoderLayer(d_model, nhead, dim_ff, dropout, batch_first=True)
        self.enc = nn.TransformerEncoder(enc_layer, num_layers)
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, out_dim)
        )

    def forward(self, x):
        z = self.inproj(x)
        z = self.pos(z)
        z = self.enc(z)
        h = z[:, -1, :]
        y = self.head(h)
        return y



In [24]:
# 8) Training & Evaluation

def dir_acc(y_true, y_pred, eps=0.0):
    mask = np.ones_like(y_pred, dtype=bool) if eps == 0 else (np.abs(y_pred) >= eps)
    if mask.sum() == 0:
        return np.nan
    return ((y_true[mask] >= 0) == (y_pred[mask] >= 0)).mean()

def metrics_multi(y_true, y_pred):
    out = {}
    names = ["2h", "4h", "6h"]
    for i, name in enumerate(names):
        yt, yp = y_true[:, i], y_pred[:, i]
        out[f"{name}_MAE"] = mean_absolute_error(yt, yp)
        out[f"{name}_RMSE"] = np.sqrt(mean_squared_error(yt, yp))
        out[f"{name}_DA"] = dir_acc(yt, yp, eps=0.0)
        out[f"{name}_DAe"] = dir_acc(yt, yp, eps=1e-4)
    return out

def evaluate(model, dl, crit):
    model.eval()
    loss_sum, n = 0.0, 0
    Ys, Ps = [], []
    with torch.no_grad():
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = crit(pred, yb)
            bs = len(yb)
            loss_sum += loss.item() * bs
            n += bs
            Ys.append(yb.cpu().numpy())
            Ps.append(pred.cpu().numpy())
    Ys = np.concatenate(Ys) if Ys else np.zeros((0, 3))
    Ps = np.concatenate(Ps) if Ps else np.zeros((0, 3))
    mets = metrics_multi(Ys, Ps)
    return loss_sum / max(n, 1), mets, Ys, Ps

def train_one(model, epochs=25, lr=1e-3, patience=6, name="MODEL"):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.HuberLoss(delta=1.0)
    best, bad = 1e9, 0
    best_path = f"best_{name}.pt"

    print(f"\n=== START TRAIN [{name}] ===")
    print("[SANITY] dataset sizes -> train={}, val={}, test={}".format(len(tr_ds), len(va_ds), len(te_ds)))
    xb, yb = next(iter(train_dl))
    print("[SANITY] one batch -> X={}, y={}".format(xb.shape, yb.shape))

    for ep in range(1, epochs + 1):
        t0 = time.time()
        model.train()
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            pred = model(xb)
            loss = crit(pred, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        tr_loss, tr_m, *_ = evaluate(model, train_dl, crit)
        va_loss, va_m, *_ = evaluate(model, val_dl, crit)
        dt = time.time() - t0

        def fmt(m):
            return " ".join([f"{k}={v:.4f}" if "DA" not in k else f"{k}={v*100:.2f}%" for k, v in m.items()])
        print(f"[{name}] Ep{ep:02d} | train LOSS={tr_loss:.6f} | val LOSS={va_loss:.6f} | "
              f"val({fmt(va_m)}) | {dt:.1f}s")

        if va_loss < best - 1e-9:
            best, bad = va_loss, 0
            torch.save(model.state_dict(), best_path)
        else:
            bad += 1
            if bad >= patience:
                print(f"[{name}] Early stopping at epoch {ep}")
                break

    model.load_state_dict(torch.load(best_path, map_location=device))
    te_loss, te_m, Yt, Yp = evaluate(model, test_dl, crit)
    print(f"[{name}][TEST] LOSS={te_loss:.6f} | test({ ' '.join([f'{k}={v:.4f}' if 'DA' not in k else f'{k}={v*100:.2f}%' for k,v in te_m.items()]) })")
    return model, te_m, (Yt, Yp)


In [25]:
# 9) Train Models

gru = RNNMulti(in_feats, hidden=64, layers=2, kind="gru", dropout=0.2, out_dim=3, bidir=False)
gru, gru_test, (Yt_gru, Yp_gru) = train_one(gru, epochs=25, lr=1e-3, patience=6, name="GRU")

lstm = RNNMulti(in_feats, hidden=64, layers=2, kind="lstm", dropout=0.2, out_dim=3, bidir=False)
lstm, lstm_test, (Yt_lstm, Yp_lstm) = train_one(lstm, epochs=25, lr=1e-3, patience=6, name="LSTM")

tcn = TCNMulti(in_feats, ch=64, depth=4, k=3, dropout=0.2, out_dim=3)
tcn, tcn_test, (Yt_tcn, Yp_tcn) = train_one(tcn, epochs=25, lr=1e-3, patience=6, name="TCN")

trans = TransEncMulti(in_feats, d_model=64, nhead=4, num_layers=2, dim_ff=128, dropout=0.1, out_dim=3)
trans, trans_test, (Yt_tr, Yp_tr) = train_one(trans, epochs=25, lr=1e-3, patience=6, name="TRANS")



=== START TRAIN [GRU] ===
[SANITY] dataset sizes -> train=14035, val=1968, test=840
[SANITY] one batch -> X=torch.Size([256, 240, 6]), y=torch.Size([256, 3])
[GRU] Ep01 | train LOSS=0.000037 | val LOSS=0.000015 | val(2h_MAE=0.0035 2h_RMSE=0.0050 2h_DA=49.19% 2h_DAe=48.96% 4h_MAE=0.0039 4h_RMSE=0.0054 4h_DA=66.36% 4h_DAe=66.56% 6h_MAE=0.0042 6h_RMSE=0.0058 6h_DA=72.21% 6h_DAe=72.44%) | 1.0s
[GRU] Ep02 | train LOSS=0.000034 | val LOSS=0.000014 | val(2h_MAE=0.0037 2h_RMSE=0.0051 2h_DA=50.30% 2h_DAe=50.36% 4h_MAE=0.0036 4h_RMSE=0.0051 4h_DA=72.46% 4h_DAe=72.91% 6h_MAE=0.0041 6h_RMSE=0.0057 6h_DA=71.80% 6h_DAe=72.06%) | 1.1s
[GRU] Ep03 | train LOSS=0.000034 | val LOSS=0.000015 | val(2h_MAE=0.0034 2h_RMSE=0.0049 2h_DA=50.36% 2h_DAe=50.26% 4h_MAE=0.0037 4h_RMSE=0.0052 4h_DA=68.95% 4h_DAe=69.50% 6h_MAE=0.0047 6h_RMSE=0.0062 6h_DA=68.50% 6h_DAe=68.83%) | 1.5s
[GRU] Ep04 | train LOSS=0.000034 | val LOSS=0.000018 | val(2h_MAE=0.0054 2h_RMSE=0.0066 2h_DA=49.85% 2h_DAe=49.87% 4h_MAE=0.0038 4h_RMSE

In [23]:
# 10) Backtest & Metrics Summary

import numpy as np
import pandas as pd

def max_drawdown(cumret):
    x = np.asarray(cumret, dtype=float)
    if x.size == 0:
        return np.nan
    peak = np.maximum.accumulate(x)
    dd = x - peak
    return float(dd.min())

def backtest_from_preds(Y_true, Y_pred, eps=0.0, tcost_bps=0.0):
    res = {}
    Hs = ["2h", "4h", "6h"]
    for i, h in enumerate(Hs):
        yt = np.asarray(Y_true[:, i], dtype=float)
        yp = np.asarray(Y_pred[:, i], dtype=float)
        if yt.size == 0:
            res[h] = {"RET": np.nan, "SR": np.nan, "MDD": np.nan, "Hit": np.nan, "Trades": 0}
            continue
        sig = np.where(np.abs(yp) >= eps, np.sign(yp), 0.0)
        trades = int(np.sum(np.abs(np.diff(sig)) > 0) + (sig[0] != 0))
        cost = trades * (tcost_bps / 10000.0)
        strat_r = sig * yt
        ret = float(np.nansum(strat_r) - cost)
        mu = np.nanmean(strat_r) if np.isfinite(strat_r).all() else np.nanmean(np.where(np.isfinite(strat_r), strat_r, 0))
        sd = np.nanstd(strat_r, ddof=1)
        sr = float(mu / sd) if (sd is not None and sd > 0) else np.nan
        equity = np.nancumsum(strat_r) - np.linspace(0, cost, num=len(strat_r))
        mdd = max_drawdown(equity)
        hit = float(((yt >= 0) == (yp >= 0)).mean())
        res[h] = {"RET": ret, "SR": sr, "MDD": mdd, "Hit": hit, "Trades": trades}
    return res

EPS = 0.0
TCOST_BPS = 0.0

bt_gru   = backtest_from_preds(Yt_gru,  Yp_gru,  eps=EPS, tcost_bps=TCOST_BPS)
bt_lstm  = backtest_from_preds(Yt_lstm, Yp_lstm, eps=EPS, tcost_bps=TCOST_BPS)
bt_tcn   = backtest_from_preds(Yt_tcn,  Yp_tcn,  eps=EPS, tcost_bps=TCOST_BPS)
bt_trans = backtest_from_preds(Yt_tr,   Yp_tr,   eps=EPS, tcost_bps=TCOST_BPS)

def flatten_bt(bt_dict, default_metric="RET"):
    flat = {}
    if not isinstance(bt_dict, dict):
        return flat
    for h, val in bt_dict.items():
        if isinstance(val, dict):
            for metric, v in val.items():
                flat[f"{h}_{metric}"] = float(v) if np.isscalar(v) and np.isfinite(v) else np.nan
        else:
            flat[f"{h}_{default_metric}"] = float(val) if np.isscalar(val) else np.nan
    return flat

bt_gru_flat   = flatten_bt(bt_gru)
bt_lstm_flat  = flatten_bt(bt_lstm)
bt_tcn_flat   = flatten_bt(bt_tcn)
bt_trans_flat = flatten_bt(bt_trans)

all_keys = sorted(set(bt_gru_flat) | set(bt_lstm_flat) | set(bt_tcn_flat) | set(bt_trans_flat))
bt_df = pd.DataFrame({
    "GRU":   [bt_gru_flat.get(k,  np.nan) for k in all_keys],
    "LSTM":  [bt_lstm_flat.get(k, np.nan) for k in all_keys],
    "TCN":   [bt_tcn_flat.get(k,  np.nan) for k in all_keys],
    "TRANS": [bt_trans_flat.get(k, np.nan) for k in all_keys],
}, index=all_keys)

display(bt_df)



,GRU,LSTM,TCN,TRANS
2h_Hit,0.497619,0.509524,0.508333,0.514286
2h_MDD,-0.163419,-0.323980,-0.286521,-0.205192
2h_RET,0.046212,-0.148741,0.059254,-0.136413
2h_SR,0.011122,-0.035820,0.014262,-0.032848
2h_Trades,27.000000,53.000000,1.000000,292.000000
4h_Hit,0.720238,0.726190,0.485714,0.646429
4h_MDD,-0.029371,-0.031747,-0.831690,-0.032604
4h_RET,2.746277,2.709680,-0.643201,2.123666
4h_SR,0.512443,0.503863,-0.107413,0.376851
4h_Trades,248.000000,248.000000,2.000000,334.000000
